# Notebook 1: Experiment 1 — Same-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on each stock's daily data and predict its own future prices.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Stocks:** TLKM, BBCA, ASII, UNVR  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()
check_gpu()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp1_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 1 - Same Stock Prediction (80/20)")
print(f"Train ratio: {TRAIN_RATIO}, Test ratio: {1-TRAIN_RATIO}")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

GPU Memory allocation: Dynamic growth up to 95% (for optimal utilization)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations

Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 1 - Same Stock Prediction (80/20)
Train ratio: 0.8, Test ratio: 0.19999999999999996


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


In [3]:
# Reload the module to get the latest changes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

GPU Memory allocation: Dynamic growth up to 95% (for optimal utilization)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


In [4]:
# ============================================================
# INTERACTIVE DATA SPLIT VISUALIZATION
# ============================================================

# Individual stock data split visualizations
print("Generating interactive data split visualizations...\n")

for stock in STOCKS:
    fig, html_file = create_interactive_data_split_visualization(
        daily_data[stock], 
        train_ratio=TRAIN_RATIO,
        stock_name=stock,
        experiment_label=EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )
    print(f"✓ {stock} split visualization saved: {html_file}")
    fig.show()

print("\n")

# Summary visualization - all stocks
print("Generating summary data split visualization for all stocks...\n")
fig_summary, html_summary = create_interactive_split_summary_visualization(
    daily_data,
    train_ratio=TRAIN_RATIO,
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}'
)
print(f"✓ Summary visualization saved: {html_summary}")
fig_summary.show()

print("\n")

# Statistics bar chart
print("Generating data split statistics bar chart...\n")
fig_stats, html_stats = create_interactive_split_bar_chart(
    daily_data,
    train_ratio=TRAIN_RATIO,
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}'
)
print(f"✓ Statistics chart saved: {html_stats}")
fig_stats.show()

print("\n✓ All data split visualizations generated successfully!")

Generating interactive data split visualizations...

✓ TLKM split visualization saved: figures/Exp1_80_20/interactive_data_split_TLKM_80_19.html


✓ BBCA split visualization saved: figures/Exp1_80_20/interactive_data_split_BBCA_80_19.html


✓ ASII split visualization saved: figures/Exp1_80_20/interactive_data_split_ASII_80_19.html


✓ UNVR split visualization saved: figures/Exp1_80_20/interactive_data_split_UNVR_80_19.html




Generating summary data split visualization for all stocks...

✓ Summary visualization saved: figures/Exp1_80_20/interactive_data_split_all_stocks_80_19.html




Generating data split statistics bar chart...

✓ Statistics chart saved: figures/Exp1_80_20/interactive_split_statistics_80_19.html



✓ All data split visualizations generated successfully!


## Data Split Visualization

## Run All Experiments

In [5]:
# ============================================================
# EXPERIMENT 1: Train and predict on same stock
# ============================================================
all_results = []
all_predictions = {}  # {stock: {model_type: (y_true, y_pred, dates)}}
all_histories = {}    # {stock: {model_type: history}}

for stock in STOCKS:
    print(f"\n############################################################")
    print(f"# STOCK: {stock}")
    print(f"############################################################")
    
    # Prepare data
    X_train, y_train, X_test, y_test, test_dates = prepare_same_stock_data(
        daily_data[stock], train_ratio=TRAIN_RATIO, lookback=LOOKBACK
    )
    print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
    
    all_predictions[stock] = {}
    all_histories[stock] = {}
    
    for model_type in MODEL_TYPES:
        exp_name = f'{EXP_LABEL}_{stock}'
        
        y_true_inv, y_pred_inv, metrics, history = train_and_evaluate(
            model_type=model_type,
            X_train=X_train, y_train=y_train,
            X_test=X_test, y_test=y_test,
            experiment_name=exp_name,
            save_dir=f'models/{EXP_LABEL}',
            epochs=EPOCHS, batch_size=BATCH_SIZE
        )
        
        # Store results
        result = {'Stock': stock, 'Model': model_type, **metrics}
        all_results.append(result)
        all_predictions[stock][model_type] = (y_true_inv, y_pred_inv, test_dates)
        all_histories[stock][model_type] = history
        
        # Plot individual prediction
        plot_actual_vs_predicted(
            test_dates, y_true_inv, y_pred_inv,
            model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )
        
        # Plot training history
        plot_training_history(
            history, model_type, stock, EXP_LABEL,
            save_dir=f'figures/{EXP_LABEL}'
        )

print("\n\nAll Experiment 1 (80/20) training complete!")



############################################################
# STOCK: TLKM
############################################################
  X_train: (4193, 1, 1), X_test: (1049, 1, 1)

Training BiLSTM for: Exp1_80_20_TLKM
  Train samples: 4193, Test samples: 1049
Epoch 1/100
56/59 [===========================>..] - ETA: 0s - loss: 0.0068
Epoch 1: val_loss improved from inf to 0.00250, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 17s 49ms/step - loss: 0.0066 - val_loss: 0.0025
Epoch 2/100
57/59 [===========================>..] - ETA: 0s - loss: 5.3754e-04
Epoch 2: val_loss improved from 0.00250 to 0.00004, saving model to models/Exp1_80_20\Exp1_80_20_TLKM_BiLSTM_best.keras
59/59 [==============================] - 1s 16ms/step - loss: 5.2531e-04 - val_loss: 3.6764e-05
Epoch 3/100
58/59 [============================>.] - ETA: 0s - loss: 1.0764e-04
Epoch 3: val_loss did not improve from 0.00004
59/59 [==========================

## Results Summary

In [6]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 1 - Same Stock Prediction (80/20)")

# Save results
results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 1 - Same Stock Prediction (80/20)
Stock  Model        MSE     RMSE      MAE  MAPE (%)       R2  Training_Time_s  Epochs_Run
 TLKM BiLSTM  3687.8449  60.7276  46.3050    1.4934 0.978413            106.6         100
 TLKM  BiGRU  5718.8463  75.6231  61.3458    1.9658 0.966524             94.6         100
 TLKM   LSTM  3462.6299  58.8441  44.5383    1.4341 0.979731             62.6         100
 TLKM    GRU 11238.9280 106.0138  93.2521    2.9475 0.934212             51.8         100
 BBCA BiLSTM 15096.8083 122.8691  92.0115    1.1294 0.986032             83.5         100
 BBCA  BiGRU 17331.8992 131.6507  97.3645    1.1709 0.983964             80.2         100
 BBCA   LSTM 49592.6525 222.6941 176.6374    2.0455 0.954117             58.5         100
 BBCA    GRU 48182.7517 219.5057 180.4994    2.1187 0.955421             50.9         100
 ASII BiLSTM  7363.4502  85.8105  62.9898    1.3328 0.980070             83.6         100
 ASII  BiGRU  6850.6175  82.7685  60.5286    1.2855 

## Visualizations

In [7]:
# ============================================================
# ALL MODELS COMPARISON PER STOCK
# ============================================================
for stock in STOCKS:
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    preds = {mt: all_predictions[stock][mt][1] for mt in MODEL_TYPES}
    
    plot_all_models_comparison(
        dates, y_true, preds, stock, EXP_LABEL,
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All comparison plots saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_TLKM_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_BBCA_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_ASII_all_models.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_UNVR_all_models.png
All comparison plots saved!


In [8]:
# ============================================================
# METRICS BAR CHARTS
# ============================================================
for metric in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_comparison_bar(
        results_df, metric, EXP_LABEL,
        group_col='Stock', save_dir=f'figures/{EXP_LABEL}'
    )

print("All metrics bar charts saved!")


  Figure saved: figures/Exp1_80_20/Exp1_80_20_MSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_RMSE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAE_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_MAPE_pct_comparison.png
  Figure saved: figures/Exp1_80_20/Exp1_80_20_R2_comparison.png
All metrics bar charts saved!


In [9]:
# ============================================================
# SUMMARY: BEST MODEL PER STOCK
# ============================================================
print("\n" + "="*60)
print("  BEST MODEL PER STOCK (by RMSE)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['RMSE'].idxmin()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")

print("\n  BEST MODEL PER STOCK (by R² Score)")
print("="*60)
for stock in STOCKS:
    stock_results = results_df[results_df['Stock'] == stock]
    best_idx = stock_results['R2'].idxmax()
    best = stock_results.loc[best_idx]
    print(f"  {stock}: {best['Model']} (R²={best['R2']:.6f}, RMSE={best['RMSE']:.4f})")



  BEST MODEL PER STOCK (by RMSE)
  TLKM: LSTM (RMSE=58.8441, R²=0.979731)
  BBCA: BiLSTM (RMSE=122.8691, R²=0.986032)
  ASII: BiGRU (RMSE=82.7685, R²=0.981458)
  UNVR: LSTM (RMSE=76.1338, R²=0.993337)

  BEST MODEL PER STOCK (by R² Score)
  TLKM: LSTM (R²=0.979731, RMSE=58.8441)
  BBCA: BiLSTM (R²=0.986032, RMSE=122.8691)
  ASII: BiGRU (R²=0.981458, RMSE=82.7685)
  UNVR: LSTM (R²=0.993337, RMSE=76.1338)


## Interactive Prediction Visualization (Plotly)
Zoom in, pan, and explore the price predictions with hover details.

In [10]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Interactive Results Dashboard
print("1. Generating Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp1(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Interactive Heatmaps for each metric
print("2. Generating Interactive Heatmaps...")
for metric in ['RMSE', 'MAE', 'R2', 'MAPE (%)']:
    try:
        fig, html = create_interactive_metrics_heatmap_exp1(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"   ✓ {metric} heatmap: {html}")
        fig.show()
    except Exception as e:
        print(f"   ⚠ Skipping {metric}: {str(e)}")

print()

# 3. Metrics Comparison Chart
print("3. Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("4. Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html4}")
fig4.show()

print("\n✓ All interactive results visualizations generated successfully!")

Generating interactive results visualizations...

1. Generating Results Dashboard...
   ✓ Saved: figures/Exp1_80_20/Exp1_80_20_results_dashboard.html



2. Generating Interactive Heatmaps...
   ✓ RMSE heatmap: figures/Exp1_80_20/Exp1_80_20_RMSE_heatmap_interactive.html


   ✓ MAE heatmap: figures/Exp1_80_20/Exp1_80_20_MAE_heatmap_interactive.html


   ✓ R2 heatmap: figures/Exp1_80_20/Exp1_80_20_R2_heatmap_interactive.html


   ✓ MAPE (%) heatmap: figures/Exp1_80_20/Exp1_80_20_MAPE (%)_heatmap_interactive.html



3. Generating Metrics Comparison Chart...
   ✓ Saved: figures/Exp1_80_20/Exp1_80_20_metrics_comparison.html



4. Generating Model Radar Chart...
   ✓ Saved: figures/Exp1_80_20/Exp1_80_20_model_radar.html



✓ All interactive results visualizations generated successfully!


## Interactive Results Visualizations

In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ACTUAL VS PREDICTED (Plotly)
# Actual Prices in RED (#FF0000), Model Predictions in Model Colors
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating interactive plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    # Create figure with all models on one chart
    fig = go.Figure()
    
    # Add ACTUAL PRICES in RED first (so it appears on top)
    fig.add_trace(
        go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=2.0),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            opacity=0.95
        )
    )
    
    # Add predictions for each model
    linestyles = {'BiLSTM': 'solid', 'BiGRU': 'dash', 'LSTM': 'dot', 'GRU': 'dashdot'}
    
    for model_type in MODEL_TYPES:
        y_pred = all_predictions[stock][model_type][1]
        
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} (Predicted)',
                mode='lines',
                line=dict(
                    color=MODEL_COLORS.get(model_type, '#999'),
                    width=2.0,
                    dash=linestyles.get(model_type, 'solid')
                ),
                hovertemplate=f'<b>{model_type} PREDICTED</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                opacity=0.85
            )
        )
    
    # Update layout with enhanced styling
    fig.update_layout(
        title=f'<b>{stock} - Actual vs Predicted Stock Prices</b><br><sub>Experiment 1 (80/20 Split)</sub>',
        xaxis_title="Date",
        yaxis_title="Close Price (IDR)",
        height=700,
        template='plotly_white',
        hovermode='x unified',
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=25, family="Times New Roman")
        ),
        font=dict(size=30, family="Times New Roman"),
        xaxis=dict(
            gridwidth=1,
            gridcolor='lightgray',
            tickfont=dict(size=30, family="Times New Roman")
        ),
        yaxis=dict(
            gridwidth=1,
            gridcolor='lightgray',
            tickfont=dict(size=30, family="Times New Roman")
        ),
        xaxis_title_font=dict(size=40, family="Times New Roman"),
        yaxis_title_font=dict(size=40, family="Times New Roman")
    )
    
    # Save as HTML
    html_file = f'figures/{EXP_LABEL}/interactive_{stock}_all_models.html'
    fig.write_html(html_file, config=PLOTLY_HTML_CONFIG)
    print(f"  ✓ Saved: {html_file}")
    fig.show()

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")
print(f"  Actual prices displayed in RED (#FF0000)")
print(f"  Font: Times New Roman, Size: 20")


Generating interactive plot for TLKM...
  ✓ Saved: figures/Exp1_80_20/interactive_TLKM_all_models.html



Generating interactive plot for BBCA...
  ✓ Saved: figures/Exp1_80_20/interactive_BBCA_all_models.html



Generating interactive plot for ASII...
  ✓ Saved: figures/Exp1_80_20/interactive_ASII_all_models.html



Generating interactive plot for UNVR...
  ✓ Saved: figures/Exp1_80_20/interactive_UNVR_all_models.html



✓ All interactive plots generated and saved!
  Location: figures/Exp1_80_20/interactive_*.html
  Actual prices displayed in RED (#FF0000)
  Font: Times New Roman, Size: 20


In [16]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE MODELS WITH BUTTONS
# Actual Prices in RED (#FF0000)
# ============================================================

for stock in STOCKS:
    print(f"\nGenerating toggle model plot for {stock}...")
    
    y_true = all_predictions[stock][MODEL_TYPES[0]][0]
    dates = all_predictions[stock][MODEL_TYPES[0]][2]
    
    fig = go.Figure()
    
    # Add actual price (always visible, in RED)
    fig.add_trace(
        go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=2.0),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True,
            opacity=0.95
        )
    )
    
    # Add predictions for each model (togglable)
    linestyles = {'BiLSTM': 'solid', 'BiGRU': 'dash', 'LSTM': 'dot', 'GRU': 'dashdot'}
    
    for model_type in MODEL_TYPES:
        y_pred = all_predictions[stock][model_type][1]
        
        fig.add_trace(
            go.Scatter(
                x=dates, y=y_pred,
                name=f'{model_type} (Predicted)',
                mode='lines',
                line=dict(
                    color=MODEL_COLORS.get(model_type, '#999'),
                    width=2.0,
                    dash=linestyles.get(model_type, 'solid')
                ),
                hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                visible=True,
                opacity=0.85
            )
        )
    
    # Create buttons for model selection
    buttons = [
        dict(
            label="All Models",
            method="update",
            args=[{"visible": [True] * (len(MODEL_TYPES) + 1)},
                  {"title": f"<b>{stock} - All Models vs Actual Price</b><br><sub>Experiment 1 (80/20 Split)</sub>"}]
        )
    ]
    
    for i, model_type in enumerate(MODEL_TYPES):
        visible = [True] + [False] * len(MODEL_TYPES)
        visible[i + 1] = True
        buttons.append(
            dict(
                label=model_type,
                method="update",
                args=[{"visible": visible},
                      {"title": f"<b>{stock} - {model_type} vs Actual Price</b><br><sub>Experiment 1 (80/20 Split)</sub>"}]
            )
        )
    
    # Update layout with buttons
    fig.update_layout(
        updatemenus=[
            dict(
                type="dropdown",
                direction="down",
                x=0.01,
                y=0.99,
                showactive=True,
                buttons=buttons,
                bgcolor="rgba(200, 200, 200, 0.9)",
                bordercolor="gray",
                borderwidth=2,
                font=dict(size=16, family="Times New Roman")
            )
        ],
        title=f"<b>{stock} - All Models vs Actual Price</b><br><sub>Experiment 1 (80/20 Split)</sub>",
        xaxis_title="Date",
        yaxis_title="Close Price (IDR)",
        template="plotly_white",
        hovermode="x unified",
        height=700,
        font=dict(size=20, family="Times New Roman"),
        xaxis=dict(
            rangeslider=dict(visible=False),
            gridwidth=1,
            gridcolor='lightgray',
            tickfont=dict(size=20, family="Times New Roman")
        ),
        yaxis=dict(
            gridwidth=1,
            gridcolor='lightgray',
            tickfont=dict(size=20, family="Times New Roman")
        ),
        legend=dict(
            x=0.99,
            y=0.99,
            xanchor="right",
            yanchor="top",
            bgcolor="rgba(255, 255, 255, 0.9)",
            bordercolor="gray",
            borderwidth=1,
            font=dict(size=18, family="Times New Roman")
        ),
        showlegend=True,
        xaxis_title_font=dict(size=25, family="Times New Roman"),
        yaxis_title_font=dict(size=25, family="Times New Roman")
    )
    
    # Save as HTML
    html_file = f'figures/{EXP_LABEL}/interactive_{stock}_toggle.html'
    fig.write_html(html_file, config=PLOTLY_HTML_CONFIG)
    print(f"  ✓ Saved: {html_file}")


Generating toggle model plot for TLKM...
  ✓ Saved: figures/Exp1_80_20/interactive_TLKM_toggle.html

Generating toggle model plot for BBCA...
  ✓ Saved: figures/Exp1_80_20/interactive_BBCA_toggle.html

Generating toggle model plot for ASII...
  ✓ Saved: figures/Exp1_80_20/interactive_ASII_toggle.html

Generating toggle model plot for UNVR...
  ✓ Saved: figures/Exp1_80_20/interactive_UNVR_toggle.html


## Case-by-Case Interactive Actual vs Predicted
One interactive Plotly chart per notebook with a dropdown that walks through every test case. Each case shows the Actual price (red) plus all four model predictions, with a metrics panel (MSE, RMSE, MAE, MAPE, R²) for that case.

In [13]:
# ============================================================
# CASE-BY-CASE INTERACTIVE ACTUAL VS PREDICTED (Experiment 1)
# ============================================================
# Cases follow the test plan: Train Stock X Daily -> Predict Stock X Daily
# for X in {TLKM, BBCA, ASII, UNVR}. All four models on the same chart.
from collections import OrderedDict

cases_dict = OrderedDict()
for stock in STOCKS:
    if stock not in all_predictions:
        continue
    available_mts = [mt for mt in MODEL_TYPES if mt in all_predictions[stock]]
    if not available_mts:
        continue
    y_true, _, dates = all_predictions[stock][available_mts[0]]
    predictions = {mt: all_predictions[stock][mt][1] for mt in available_mts}

    metrics = {}
    for mt in MODEL_TYPES:
        row = results_df[(results_df['Stock'] == stock) &
                         (results_df['Model'] == mt)]
        if not row.empty:
            r = row.iloc[0]
            metrics[mt] = {
                'MSE'      : r.get('MSE'),
                'RMSE'     : r.get('RMSE'),
                'MAE'      : r.get('MAE'),
                'MAPE (%)' : r.get('MAPE (%)'),
                'R2'       : r.get('R2'),
            }

    cases_dict[stock] = {
        'description' : f'Train {stock} Daily → Predict {stock} Daily',
        'dates'       : dates,
        'y_true'      : y_true,
        'predictions' : predictions,
        'metrics'     : metrics,
    }

ratio_pretty = RATIO_LABEL.replace('_', '/')
fig_case, html_case = create_case_by_case_actual_vs_predicted(
    cases_dict,
    experiment_title=f'Experiment 1: Same-Stock Prediction ({ratio_pretty})',
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}',
)
print(f"✓ Case-by-case visualization saved: {html_case}")
print(f"  Cases: {list(cases_dict.keys())}")
fig_case.show()


✓ Case-by-case visualization saved: figures/Exp1_80_20/Exp1_80_20_case_by_case_actual_vs_predicted.html
  Cases: ['TLKM', 'BBCA', 'ASII', 'UNVR']
